# Practice 2 — Team training on Google Colab
ResNet-18, VGG-16, DenseNet-121 và MobileNetV4 Conv Small. Checkpoint và TensorBoard được lưu trên Google Drive sau mỗi epoch.

## 1. Bật GPU
Chọn **Runtime → Change runtime type → T4 GPU**, sau đó chạy các cell từ trên xuống.

In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/nguyen-hai-anh06/deep-learning-lab2.git'
PROJECT_DIR = '/content/deep-learning-lab2'
if os.path.isdir(os.path.join(PROJECT_DIR, '.git')):
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run(['python', '-m', 'pytest', '-q', 'tests/test_data.py'], check=True)


In [ ]:
import torch
assert torch.cuda.is_available(), 'Hãy bật GPU runtime trước khi train.'
print('GPU:', torch.cuda.get_device_name(0))

# Chỉ đổi ROLE theo người đang chạy: trainer_a hoặc trainer_b
ROLE = 'trainer_a'
CONFIG = f'configs/{ROLE}.json'
EXPERIMENT_ROOT = '/content/drive/MyDrive/Lab2_Experiments'
DATA_DIR = '/content/cifar10_data'
print('Assignment:', CONFIG)
print('Persistent output:', EXPERIMENT_ROOT)


## 2. Smoke test
Chạy tập nhỏ để phát hiện lỗi trước. Kết quả thử nằm trong /content, không trộn với thí nghiệm thật.

In [ ]:
subprocess.run([
    'python', 'train.py', '--model', 'mobilenetv4_conv_small',
    '--strategy', 'freeze', '--epochs', '1', '--subset', '200',
    '--member-id', ROLE, '--run-id', 'smoke_test',
    '--output-root', '/content/lab2_smoke', '--data-dir', DATA_DIR
], check=True)


## 3. Chạy toàn bộ phần được giao
Nếu Colab ngắt, chạy lại notebook và cell này. Script tự tìm last.pt, tiếp tục từ epoch kế tiếp và bỏ qua run đã hoàn thành.

In [ ]:
subprocess.run([
    'python', 'run_assignment.py', '--config', CONFIG,
    '--output-root', EXPERIMENT_ROOT, '--data-dir', DATA_DIR
], check=True)


## 4. Xem TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/Lab2_Experiments


## 5. Chỉ người quản lý chạy đánh giá cuối
Chạy sau khi đã gom đủ run của hai trainer vào Lab2_Experiments. Script chọn run tốt nhất bằng validation accuracy rồi mới dùng test set.

In [ ]:
# Tổng hợp kết quả validation từ cả 2 trainer
subprocess.run([
    'python', 'collect_results.py', '--experiments-root', EXPERIMENT_ROOT
], check=True)

# Đánh giá 1 lần duy nhất trên test set cho 4 model tốt nhất
subprocess.run([
    'python', 'run_evaluation.py', '--experiments-root', EXPERIMENT_ROOT,
    '--output-dir', os.path.join(EXPERIMENT_ROOT, 'final_evaluation'),
    '--data-dir', DATA_DIR
], check=True)


## 6. Trực quan hóa toàn bộ biểu đồ (Loss, Accuracy, Parameters, Training Time)
Tự động xuất 7 biểu đồ chất lượng cao và báo cáo thực nghiệm trực tiếp ngay trong Colab.

In [ ]:
VIZ_DIR = os.path.join(EXPERIMENT_ROOT, 'visualizations')
subprocess.run([
    'python', 'visualize_experiments.py', '--experiments-root', EXPERIMENT_ROOT,
    '--output-dir', VIZ_DIR
], check=True)

from IPython.display import Image, display
import glob
for img_path in sorted(glob.glob(os.path.join(VIZ_DIR, '*.png'))):
    print(f'=== {os.path.basename(img_path)} ===')
    display(Image(filename=img_path))
